# Training CNN and BERT for IMDb Sentiment Analysis

This notebook handles the deep learning part of our project. We're training two models here:
- A CNN based on Kim (2014) with GloVe word vectors
- BERT (bert-base-uncased) fine-tuned for binary sentiment

Both get evaluated on the full IMDb test set and on our three challenging slices
(long reviews, negation-heavy, mixed sentiment). We also run them on SST-2 as
a secondary benchmark for the document-vs-phrase-level comparison.

Make sure you're on a GPU runtime before running this — go to Runtime > Change runtime type > T4 or A100.

## Setup

In [ ]:
!pip install -q torch torchvision transformers datasets scikit-learn pandas numpy scipy

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Downloading the datasets

We need IMDb (from Stanford), GloVe 6B embeddings for the CNN, and SST-2 gets pulled later via HuggingFace.

In [ ]:
import os, re, glob, time, zipfile, urllib.request
import numpy as np
import pandas as pd
from collections import Counter

# ── Download IMDb ──
if not os.path.isdir('aclImdb'):
    print('Downloading IMDb dataset ...')
    !wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    !tar -xzf aclImdb_v1.tar.gz
    !rm aclImdb_v1.tar.gz
    print('Done.')
else:
    print('IMDb already present.')

# ── Download GloVe ──
GLOVE_PATH = 'glove.6B.100d.txt'
if not os.path.exists(GLOVE_PATH):
    print('Downloading GloVe embeddings (~862 MB) ...')
    urllib.request.urlretrieve('https://nlp.stanford.edu/data/glove.6B.zip', 'glove.6B.zip')
    with zipfile.ZipFile('glove.6B.zip', 'r') as zf:
        zf.extract('glove.6B.100d.txt')
    os.remove('glove.6B.zip')
    print('Done.')
else:
    print('GloVe already present.')

print('\nAll data ready.')

## Helper functions

Same preprocessing pipeline and evaluation code we use across all our models — kept consistent so the comparison is fair. The challenging slices are:
- **Long reviews**: >= 500 words (tests whether models handle distributed sentiment)
- **Negation-heavy**: >= 2 negation cues like "not", "never", etc.
- **Mixed sentiment**: >= 2 positive AND >= 2 negative lexicon words (the hardest slice)

In [ ]:
# ── Lexicons ──
POS_WORDS = set('good great excellent amazing wonderful fantastic brilliant awesome loved love enjoyable enjoy best perfect beautiful fun funny hilarious superb outstanding masterpiece memorable touching heartwarming powerful charming delightful impressive engaging gripping thrilling stunning magnificent'.split())
NEG_WORDS = set('bad terrible awful horrible worst boring dull stupid poor waste wasted disappointing disappointed hate hated hates annoying pathetic lame ridiculous worse lousy mediocre forgettable painful laughable weak predictable cliched unwatchable tedious bland flat sloppy uninspired cringe cringy nonsense'.split())
NEGATIONS = set('not no never none nothing neither nor hardly scarcely barely without cannot cant couldnt wasnt werent isnt arent dont didnt doesnt wont wouldnt shouldnt'.split())

# ── Load IMDb ──
def load_imdb_split(split):
    texts, labels = [], []
    for label_name, label_id in [('pos', 1), ('neg', 0)]:
        for path in sorted(glob.glob(os.path.join('aclImdb', split, label_name, '*.txt'))):
            with open(path, 'r', encoding='utf-8') as f:
                texts.append(f.read())
            labels.append(label_id)
    return texts, labels

# ── Load SST-2 ──
def load_sst2():
    from datasets import load_dataset
    ds = load_dataset('glue', 'sst2')
    X_train = [ex['sentence'] for ex in ds['train']]
    y_train = [ex['label'] for ex in ds['train']]
    X_test  = [ex['sentence'] for ex in ds['validation']]
    y_test  = [ex['label'] for ex in ds['validation']]
    return X_train, y_train, X_test, y_test

# ── Preprocessing ──
_HTML_TAG = re.compile(r'<[^>]+>')
_NON_ALPHA = re.compile(r'[^a-z\s]')

def clean(text):
    text = _HTML_TAG.sub(' ', text)
    text = text.lower()
    text = _NON_ALPHA.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# ── Challenging slices ──
def build_slices(cleaned_texts):
    long_idx, neg_idx, mixed_idx = [], [], []
    for i, t in enumerate(cleaned_texts):
        words = t.split()
        if len(words) >= 500:
            long_idx.append(i)
        if sum(w in NEGATIONS for w in words) >= 2:
            neg_idx.append(i)
        pos_hits = sum(w in POS_WORDS for w in words)
        neg_hits = sum(w in NEG_WORDS for w in words)
        if pos_hits >= 2 and neg_hits >= 2:
            mixed_idx.append(i)
    return {
        'Full test set': list(range(len(cleaned_texts))),
        f'Long reviews (>=500 words) [n={len(long_idx)}]': long_idx,
        f'Negation-heavy (>=2 cues) [n={len(neg_idx)}]': neg_idx,
        f'Mixed sentiment (pos&neg>=2) [n={len(mixed_idx)}]': mixed_idx,
    }

# ── Evaluation ──
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_on_slices(y_true, y_pred, slices):
    results = []
    for name, idx in slices.items():
        idx = np.array(idx)
        yt, yp = y_true[idx], y_pred[idx]
        results.append({
            'slice': name, 'n': len(idx),
            'acc': accuracy_score(yt, yp),
            'prec': precision_score(yt, yp, zero_division=0),
            'rec': recall_score(yt, yp, zero_division=0),
            'f1': f1_score(yt, yp, zero_division=0),
        })
    return results

def print_results(results, model_name):
    df = pd.DataFrame(results)
    df.insert(0, 'model', model_name)
    print(df.to_string(index=False,
          formatters={'acc':'{:.4f}'.format, 'prec':'{:.4f}'.format,
                      'rec':'{:.4f}'.format, 'f1':'{:.4f}'.format}))
    return df

print('Utilities loaded.')

## Load IMDb and build the slice indices

In [ ]:
print('Loading IMDb ...')
t0 = time.time()
X_train_text, y_train = load_imdb_split('train')
X_test_text, y_test = load_imdb_split('test')
print(f'  train: {len(X_train_text):,}  test: {len(X_test_text):,}  ({time.time()-t0:.1f}s)')

print('Cleaning ...')
X_train_clean = [clean(t) for t in X_train_text]
X_test_clean  = [clean(t) for t in X_test_text]

y_train_np = np.array(y_train)
y_test_np  = np.array(y_test)

# Build slices (used by both CNN and BERT)
slices = build_slices(X_test_clean)
for k, v in slices.items():
    print(f'  {k:45s}  n = {len(v):>5}')

---
# Part 1: CNN (Kim 2014)

Using the multi-filter architecture from "Convolutional Neural Networks for Sentence Classification" (Kim, 2014). Filter widths of 3, 4, 5 over GloVe 100d embeddings, with max-over-time pooling. This is a well-established baseline for text classification that should do better than bag-of-words since it captures local phrase patterns.

## Vocabulary + GloVe embeddings

Building the vocab from training data and loading pre-trained GloVe vectors. Words not in GloVe get random initialization.

In [ ]:
MAX_VOCAB = 50000
EMBED_DIM = 100
MAX_LEN = 500

# Build vocab from training data
counter = Counter()
for text in X_train_clean:
    counter.update(text.split())

vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.most_common(MAX_VOCAB))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
VOCAB_SIZE = len(vocab)
print(f'Vocabulary size: {VOCAB_SIZE}')

def text_to_indices(text, max_len):
    words = text.split()[:max_len]
    indices = [vocab.get(w, 1) for w in words]
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    return indices

# Load GloVe
print(f'Loading GloVe from {GLOVE_PATH} ...')
glove = {}
with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip().split(' ')
        word = parts[0]
        if word in vocab:
            glove[word] = np.array(parts[1:], dtype=np.float32)

embedding_matrix = np.random.uniform(-0.25, 0.25, (VOCAB_SIZE, EMBED_DIM)).astype(np.float32)
embedding_matrix[0] = 0  # PAD
hits = 0
for word, idx in vocab.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]
        hits += 1
print(f'GloVe coverage: {hits}/{VOCAB_SIZE} ({hits/VOCAB_SIZE*100:.1f}%)')
del glove  # free memory

## CNN architecture and training

10 epochs — GloVe gives us a good starting point so convergence is fast. I tried 5 initially but 10 gave slightly better results without overfitting.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

class IMDbDataset(Dataset):
    def __init__(self, texts, labels, max_len):
        self.data = [text_to_indices(t, max_len) for t in texts]
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(self.data[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.float32))

class TextCNN(nn.Module):
    """Kim (2014) multi-filter CNN for text classification."""
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes,
                 dropout, pretrained_embeddings=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained_embeddings))
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, fs) for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        conv_outs = [torch.relu(conv(x)).max(dim=2).values for conv in self.convs]
        x = torch.cat(conv_outs, dim=1)
        x = self.dropout(x)
        return self.fc(x).squeeze(1)

# Hyperparameters
CNN_EPOCHS = 10
CNN_BATCH  = 64
CNN_LR     = 1e-3
NUM_FILTERS = 128
FILTER_SIZES = [3, 4, 5]
DROPOUT = 0.5

train_ds = IMDbDataset(X_train_clean, y_train, MAX_LEN)
test_ds  = IMDbDataset(X_test_clean, y_test, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=CNN_BATCH, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=CNN_BATCH, shuffle=False)

cnn_model = TextCNN(VOCAB_SIZE, EMBED_DIM, NUM_FILTERS, FILTER_SIZES,
                    DROPOUT, embedding_matrix).to(DEVICE)
print(f'CNN parameters: {sum(p.numel() for p in cnn_model.parameters()):,}')

optimizer = torch.optim.Adam(cnn_model.parameters(), lr=CNN_LR)
criterion = nn.BCEWithLogitsLoss()

print(f'\nTraining CNN for {CNN_EPOCHS} epochs ...')
for epoch in range(1, CNN_EPOCHS + 1):
    cnn_model.train()
    total_loss = correct = total = 0
    t0 = time.time()
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        logits = cnn_model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bx.size(0)
        correct += ((logits > 0).long() == by.long()).sum().item()
        total += bx.size(0)

    cnn_model.eval()
    vc = vt = 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            preds = (cnn_model(bx) > 0).long()
            vc += (preds == by.long()).sum().item()
            vt += bx.size(0)

    print(f'  Epoch {epoch}/{CNN_EPOCHS}  loss={total_loss/total:.4f}  '
          f'train_acc={correct/total:.4f}  val_acc={vc/vt:.4f}  ({time.time()-t0:.1f}s)')

## CNN results on IMDb slices

In [ ]:
cnn_model.eval()
all_preds = []
with torch.no_grad():
    for bx, _ in test_loader:
        bx = bx.to(DEVICE)
        preds = (cnn_model(bx) > 0).long()
        all_preds.extend(preds.cpu().numpy())

cnn_preds = np.array(all_preds)
cnn_results = evaluate_on_slices(y_test_np, cnn_preds, slices)

print('=== CNN IMDb RESULTS ===')
cnn_df = print_results(cnn_results, 'CNN (GloVe + Kim2014)')
cnn_df.to_csv('02_cnn_results.csv', index=False)
print('\nSaved to 02_cnn_results.csv')

## CNN on SST-2

Training a separate CNN on SST-2 to see how the same architecture performs on shorter, sentence-level reviews. Using a shorter max_len (64 tokens) since SST-2 sentences are much shorter than IMDb reviews.

In [ ]:
print('Loading SST-2 ...')
X_train_sst, y_train_sst, X_test_sst, y_test_sst = load_sst2()
X_train_sst_clean = [clean(t) for t in X_train_sst]
X_test_sst_clean  = [clean(t) for t in X_test_sst]

SST_MAX_LEN = 64
sst_train_ds = IMDbDataset(X_train_sst_clean, y_train_sst, SST_MAX_LEN)
sst_test_ds  = IMDbDataset(X_test_sst_clean, y_test_sst, SST_MAX_LEN)
sst_train_loader = DataLoader(sst_train_ds, batch_size=CNN_BATCH, shuffle=True)
sst_test_loader  = DataLoader(sst_test_ds, batch_size=CNN_BATCH, shuffle=False)

# Fresh CNN for SST-2
cnn_sst = TextCNN(VOCAB_SIZE, EMBED_DIM, NUM_FILTERS, FILTER_SIZES,
                  DROPOUT, embedding_matrix).to(DEVICE)
sst_opt = torch.optim.Adam(cnn_sst.parameters(), lr=CNN_LR)

print(f'Training CNN on SST-2 for {CNN_EPOCHS} epochs ...')
for epoch in range(1, CNN_EPOCHS + 1):
    cnn_sst.train()
    total_loss = correct = total = 0
    t0 = time.time()
    for bx, by in sst_train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        sst_opt.zero_grad()
        logits = cnn_sst(bx)
        loss = criterion(logits, by)
        loss.backward()
        sst_opt.step()
        total_loss += loss.item() * bx.size(0)
        correct += ((logits > 0).long() == by.long()).sum().item()
        total += bx.size(0)

    cnn_sst.eval()
    vc = vt = 0
    with torch.no_grad():
        for bx, by in sst_test_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            vc += ((cnn_sst(bx) > 0).long() == by.long()).sum().item()
            vt += bx.size(0)
    print(f'  Epoch {epoch}/{CNN_EPOCHS}  loss={total_loss/total:.4f}  '
          f'train_acc={correct/total:.4f}  val_acc={vc/vt:.4f}  ({time.time()-t0:.1f}s)')

# Final SST-2 eval
cnn_sst.eval()
sst_preds = []
with torch.no_grad():
    for bx, _ in sst_test_loader:
        bx = bx.to(DEVICE)
        sst_preds.extend((cnn_sst(bx) > 0).long().cpu().numpy())

y_sst = np.array(y_test_sst)
yp_sst = np.array(sst_preds)
sst_res = [{'dataset': 'SST-2', 'model': 'CNN (GloVe + Kim2014)', 'n': len(y_sst),
            'acc': accuracy_score(y_sst, yp_sst), 'prec': precision_score(y_sst, yp_sst),
            'rec': recall_score(y_sst, yp_sst), 'f1': f1_score(y_sst, yp_sst)}]
sst_df = pd.DataFrame(sst_res)
print('\n=== CNN SST-2 RESULTS ===')
print(sst_df.to_string(index=False,
      formatters={'acc':'{:.4f}'.format,'prec':'{:.4f}'.format,'rec':'{:.4f}'.format,'f1':'{:.4f}'.format}))
sst_df.to_csv('02_cnn_sst2_results.csv', index=False)
print('Saved to 02_cnn_sst2_results.csv')

---
# Part 2: BERT Fine-tuning

Now the big one. BERT already has pre-trained contextual representations so we just need to fine-tune the classification head (and the whole model) on our sentiment data. Following the standard approach from Devlin et al. — AdamW with linear warmup, small learning rate (2e-5), and 4 epochs. BERT's max input is 512 tokens, so longer IMDb reviews will get truncated (this is a known limitation we discuss in the report).

## Tokenizer and data loading

Important: we feed BERT the raw text, not our cleaned version. BERT's WordPiece tokenizer handles its own preprocessing (casing, subwords, etc.).

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup

MODEL_NAME = 'bert-base-uncased'
BERT_MAX_LEN = 512
BERT_EPOCHS = 4
BERT_BATCH = 16
BERT_LR = 2e-5
GRAD_ACCUM = 1

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

bert_train_ds = BertDataset(X_train_text, y_train, tokenizer, BERT_MAX_LEN)
bert_test_ds  = BertDataset(X_test_text, y_test, tokenizer, BERT_MAX_LEN)
bert_train_loader = DataLoader(bert_train_ds, batch_size=BERT_BATCH, shuffle=True)
bert_test_loader  = DataLoader(bert_test_ds, batch_size=BERT_BATCH, shuffle=False)

print(f'Train batches: {len(bert_train_loader)}  Test batches: {len(bert_test_loader)}')

## Training BERT on IMDb

This takes a while (~30 min on T4, ~12 min on A100). 4 epochs following the Devlin et al. recommendation of 2-4 epochs for fine-tuning.

In [ ]:
print(f'Loading {MODEL_NAME} ...')
bert_model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in bert_model.parameters()):,}')

bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps = (len(bert_train_loader) // GRAD_ACCUM) * BERT_EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(bert_optimizer, warmup_steps, total_steps)

print(f'\nTraining BERT for {BERT_EPOCHS} epochs ({total_steps} steps, {warmup_steps} warmup) ...')
for epoch in range(1, BERT_EPOCHS + 1):
    bert_model.train()
    total_loss = correct = total = 0
    t0 = time.time()
    bert_optimizer.zero_grad()

    for step, batch in enumerate(bert_train_loader, 1):
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        out = bert_model(input_ids=ids, attention_mask=mask, labels=labels)
        loss = out.loss / GRAD_ACCUM
        loss.backward()

        total_loss += out.loss.item() * ids.size(0)
        correct += (out.logits.argmax(1) == labels).sum().item()
        total += ids.size(0)

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
            bert_optimizer.step()
            scheduler.step()
            bert_optimizer.zero_grad()

        if step % 200 == 0:
            print(f'    step {step}/{len(bert_train_loader)}  '
                  f'loss={total_loss/total:.4f}  acc={correct/total:.4f}')

    # Validation
    bert_model.eval()
    vc = vt = 0
    with torch.no_grad():
        for batch in bert_test_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            out = bert_model(input_ids=ids, attention_mask=mask)
            vc += (out.logits.argmax(1) == labels).sum().item()
            vt += ids.size(0)

    print(f'  Epoch {epoch}/{BERT_EPOCHS}  loss={total_loss/total:.4f}  '
          f'train_acc={correct/total:.4f}  val_acc={vc/vt:.4f}  ({time.time()-t0:.1f}s)')

## BERT results on IMDb slices

This is where we expect the biggest improvement over baselines — especially on negation-heavy and mixed-sentiment reviews, since BERT's self-attention can actually capture long-range dependencies between negation cues and the words they modify.

In [ ]:
bert_model.eval()
all_preds = []
with torch.no_grad():
    for batch in bert_test_loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        preds = bert_model(input_ids=ids, attention_mask=mask).logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())

bert_preds = np.array(all_preds)
bert_results = evaluate_on_slices(y_test_np, bert_preds, slices)

print('=== BERT IMDb RESULTS ===')
bert_df = print_results(bert_results, f'BERT ({MODEL_NAME})')
bert_df.to_csv('03_bert_results.csv', index=False)
print('\nSaved to 03_bert_results.csv')

## BERT on SST-2

Same deal as the CNN — fresh BERT fine-tuned on SST-2 separately. Using 128 max tokens since SST-2 sentences are short.

In [ ]:
SST_BERT_MAX_LEN = 128

sst_bert_train_ds = BertDataset(X_train_sst, y_train_sst, tokenizer, SST_BERT_MAX_LEN)
sst_bert_test_ds  = BertDataset(X_test_sst, y_test_sst, tokenizer, SST_BERT_MAX_LEN)
sst_bert_train_loader = DataLoader(sst_bert_train_ds, batch_size=BERT_BATCH, shuffle=True)
sst_bert_test_loader  = DataLoader(sst_bert_test_ds, batch_size=BERT_BATCH, shuffle=False)

# Fresh BERT for SST-2
print(f'Loading fresh {MODEL_NAME} for SST-2 ...')
bert_sst = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
sst_bert_opt = torch.optim.AdamW(bert_sst.parameters(), lr=BERT_LR, weight_decay=0.01)
sst_total_steps = (len(sst_bert_train_loader) // GRAD_ACCUM) * BERT_EPOCHS
sst_scheduler = get_linear_schedule_with_warmup(
    sst_bert_opt, int(sst_total_steps * 0.1), sst_total_steps)

print(f'Training BERT on SST-2 for {BERT_EPOCHS} epochs ...')
for epoch in range(1, BERT_EPOCHS + 1):
    bert_sst.train()
    total_loss = correct = total = 0
    t0 = time.time()
    sst_bert_opt.zero_grad()

    for step, batch in enumerate(sst_bert_train_loader, 1):
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        out = bert_sst(input_ids=ids, attention_mask=mask, labels=labels)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        total_loss += out.loss.item() * ids.size(0)
        correct += (out.logits.argmax(1) == labels).sum().item()
        total += ids.size(0)
        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(bert_sst.parameters(), 1.0)
            sst_bert_opt.step()
            sst_scheduler.step()
            sst_bert_opt.zero_grad()

    bert_sst.eval()
    vc = vt = 0
    with torch.no_grad():
        for batch in sst_bert_test_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            vc += (bert_sst(input_ids=ids, attention_mask=mask).logits.argmax(1) == labels).sum().item()
            vt += ids.size(0)
    print(f'  Epoch {epoch}/{BERT_EPOCHS}  loss={total_loss/total:.4f}  '
          f'train_acc={correct/total:.4f}  val_acc={vc/vt:.4f}  ({time.time()-t0:.1f}s)')

# Final eval
bert_sst.eval()
sst_preds = []
with torch.no_grad():
    for batch in sst_bert_test_loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        sst_preds.extend(bert_sst(input_ids=ids, attention_mask=mask).logits.argmax(1).cpu().numpy())

y_sst = np.array(y_test_sst)
yp_sst = np.array(sst_preds)
sst_res = [{'dataset': 'SST-2', 'model': f'BERT ({MODEL_NAME})', 'n': len(y_sst),
            'acc': accuracy_score(y_sst, yp_sst), 'prec': precision_score(y_sst, yp_sst),
            'rec': recall_score(y_sst, yp_sst), 'f1': f1_score(y_sst, yp_sst)}]
sst_df = pd.DataFrame(sst_res)
print('\n=== BERT SST-2 RESULTS ===')
print(sst_df.to_string(index=False,
      formatters={'acc':'{:.4f}'.format,'prec':'{:.4f}'.format,'rec':'{:.4f}'.format,'f1':'{:.4f}'.format}))
sst_df.to_csv('03_bert_sst2_results.csv', index=False)
print('Saved to 03_bert_sst2_results.csv')

---
## All results side by side

Combining CNN and BERT results to see the full picture.

In [ ]:
print('='*70)
print('COMBINED IMDb RESULTS — CNN vs BERT on all slices')
print('='*70)
combined = pd.concat([cnn_df, bert_df], ignore_index=True)
print(combined.to_string(index=False,
      formatters={'acc':'{:.4f}'.format,'prec':'{:.4f}'.format,
                  'rec':'{:.4f}'.format,'f1':'{:.4f}'.format}))

combined.to_csv('combined_cnn_bert_results.csv', index=False)
print('\nSaved to combined_cnn_bert_results.csv')

## Download the result CSVs

Grab these and put them in the `code/` folder of the repo.

In [ ]:
from google.colab import files

for f in ['02_cnn_results.csv', '02_cnn_sst2_results.csv',
          '03_bert_results.csv', '03_bert_sst2_results.csv',
          'combined_cnn_bert_results.csv']:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')